## Train the CY metric, HYM bundle metrics and harmonic-form networks

Produces `model2_run_*.pkl` (trained networks + point cloud), which the matter-field and axion-coupling
notebooks load.

Tetra-quadric, defining polynomial `psi0=2, psi1=1`.  The two matter bundles are `L5=O(2,1,0,-2)` (the
10s) and `O(0,1,-2,0)` (the 5-bar); all reference forms are type-I.  The metric is trained at the D-flat
Kahler point `kmoduli=t` and orbit-averaged over the freely-acting `Z2` (flip of the 0-coordinate on
every P1).

> Heavy compute — run once to produce `model2_run_*.pkl`.

In [1]:
import numpy as np
import os, pickle, itertools, datetime
import jax, jax.numpy as jnp, jax.nn as jnn
import equinox as eqx, optax
from tqdm import tqdm
import matplotlib.pyplot as plt
from cymetric.pointgen.pointgen_mathematica import PointGeneratorMathematica
from cymetric.jax.models.models import PhiFSModel
from cymetric.jax.models.helper import train_model
from cymetric.jax.models.helper import prepare_basis as jax_prepare_basis
from cymetric.jax.models.callbacks import SigmaCallback

def kappa_features(z):
    xR = z[:8];  xI = z[8:]
    mag2      = xR**2 + xI**2
    pair_mag2 = mag2.reshape(4, 2).sum(axis=1)
    kap       = jnp.sqrt(jnp.repeat(pair_mag2, 2))
    xR = xR / kap;  xI = xI / kap
    a = jnp.array([0, 2, 4, 6]);  b = jnp.array([1, 3, 5, 7])
    feats = jnp.stack([xR[a]*xR[b] + xI[a]*xI[b], xR[a]*xI[b] - xI[a]*xR[b],
                       xR[a]**2 + xI[a]**2, xR[b]**2 + xI[b]**2], axis=1)
    return feats.reshape(-1)

# --- Z2 (model 2): a single generator, the projective flip _g1.  OUTPUT-averaged. ---
_G1IDX = jnp.array([1, 3, 5, 7, 9, 11, 13, 15])
def _g1(z): return z.at[_G1IDX].multiply(-1.0)
class OrbitAvg(eqx.Module):
    net: eqx.Module
    def __call__(self, x):
        return 0.5 * ( self.net(x) + self.net(_g1(x)) )        # Z2, not Z2xZ2
print("Z2 orbit-average + features defined.")

/opt/homebrew/Cellar/python@3.14/3.14.1/Frameworks/Python.framework/Versions/3.14/lib/python3.14/importlib/__init__.py:88: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  return _bootstrap._gcd_import(name[level:], package, level)


Z2 orbit-average + features defined.


## Kahler point (D-flat) and the model-2 constants

Specify one shape modulus `t3`, solve `t1,t4` on the D-flat cone, check the Kahler cone.  This `t` is the
metric net's `kmoduli`.

In [2]:
key = jax.random.PRNGKey(42)
psi0, psi1 = 2.0, 1.0                      # same complex structure as the tetra-quadric
ambient = np.array([1, 1, 1, 1])
alpha   = [1., 0., 1., 0., 0.]
combinations = list(itertools.product([0, 1], repeat=4))

# ---- D-flat Kahler point: specify t3, solve t1,t4, check the cone ----
t2 = 1.0
t3 = 3.0                                    # <-- the one shape modulus (needs t3>2)
t1 = (t3 - 2.0)*(t3 + 1.0)                  # mu(L4)=0 at t2=1
t4 = t1 / t3                                # mu(L1)=0 : t1 t2 = t3 t4
kmoduli = np.array([t1, t2, t3, t4], dtype=float)
assert np.all(kmoduli > 0), f"NOT in Kahler cone: {kmoduli}"
print("kmoduli (D-flat point) =", kmoduli)

# ---- matter bundles (the two the axion couples to) ----
K_L5 = np.array([2., 1., 0., -2.])         # the 10s
K_FB = np.array([0., 1., -2., 0.])         # the 5-bar
hym_names   = ['L5', 'fb']
hym_charges = [K_L5, K_FB]

n_cy_pts = 500000
dirname  = 'points_TQ_model2_500k'          # own dataset (kmoduli = D-flat point)

kmoduli (D-flat point) = [4.         1.         3.         1.33333333]


In [3]:
# Defining polynomial (identical to the tetra-quadric CS)
_mons, _cfs = [], []
for c in combinations:
    _mons.append([2*(1-c[0]),2*c[0], 2*(1-c[1]),2*c[1], 2*(1-c[2]),2*c[2], 2*(1-c[3]),2*c[3]])
    _cfs.append(1. if sum(c)%2==0 else psi0)
_mons.append([1,1,1,1,1,1,1,1]); _cfs.append(psi1)
monomials    = np.array(_mons, dtype=np.int64)
coefficients = np.array(_cfs)

pg = PointGeneratorMathematica([monomials],[coefficients], kmoduli, ambient)
basis_path = os.path.join(dirname, 'basis.pickle')
if not os.path.exists(basis_path):
    print(f"Generating {n_cy_pts} points at the D-flat kmoduli ...")
    kappa = pg.prepare_dataset(n_cy_pts, dirname); pg.prepare_basis(dirname, kappa)
with open(basis_path,'rb') as f: BASIS_raw = pickle.load(f)
BASIS = jax_prepare_basis(BASIS_raw)
data = np.load(os.path.join(dirname,'dataset.npz'))
X_train,y_train,X_val,y_val = data['X_train'],data['y_train'],data['X_val'],data['y_val']
n_train = X_train.shape[0]
print("X_train",X_train.shape,"X_val",X_val.shape)

X_train (450000, 16) X_val (50000, 16)


## Train the CY metric (Z2-averaged, at `kmoduli=t`)

In [4]:
keys = jax.random.split(key, 8); key = keys[0]
metric_nn = eqx.nn.Sequential([
    eqx.nn.Lambda(kappa_features),
    eqx.nn.Linear(16,128,key=keys[1]), eqx.nn.Lambda(jnn.gelu),
    eqx.nn.Linear(128,128,key=keys[2]), eqx.nn.Lambda(jnn.gelu),
    eqx.nn.Linear(128,128,key=keys[3]), eqx.nn.Lambda(jnn.gelu),
    eqx.nn.Linear(128,1,use_bias=False,key=keys[4]),
])
metric_model = PhiFSModel(OrbitAvg(metric_nn), BASIS, alpha=alpha)
metric_model = eqx.tree_at(lambda m: m.learn_volk,       metric_model, replace=False)
metric_model = eqx.tree_at(lambda m: m.learn_transition, metric_model, replace=False)
metric_model, hist = train_model(metric_model, data, optimizer=optax.adam(1e-3),
                                 epochs=25, callbacks=[SigmaCallback((X_val,y_val))],
                                 batch_sizes=(128,256))


Epoch  1/25


100%|██████████| 1758/1758 [01:07<00:00, 25.97it/s]


 - General loss phase: 0.128623
 - Volume loss phase:  0.054062
 - Sigma measure val:      0.0912

Epoch  2/25


100%|██████████| 1758/1758 [01:08<00:00, 25.71it/s]


 - General loss phase: 0.040616
 - Volume loss phase:  0.030286
 - Sigma measure val:      0.0635

Epoch  3/25


...

 - General loss phase: 0.008035
 - Volume loss phase:  0.005825
 - Sigma measure val:      0.0132

Epoch 24/25


100%|██████████| 1758/1758 [01:08<00:00, 25.80it/s]


 - General loss phase: 0.007901
 - Volume loss phase:  0.005666
 - Sigma measure val:      0.0138

Epoch 25/25


100%|██████████| 1758/1758 [01:08<00:00, 25.74it/s]


 - General loss phase: 0.007765
 - Volume loss phase:  0.005591
 - Sigma measure val:      0.0123


## HYM sources and training for `L5`, `fb`

In [5]:
@jax.jit
def compute_chunk(Xc):
    g = metric_model(Xc); J = metric_model.pullbacks(Xc); gi = jnp.linalg.inv(g)
    xc = (Xc[:,:8]+1j*Xc[:,8:]).astype(jnp.complex64)
    return gi, J, xc
def gref_tr(gi, J, xc, i):
    Ji=J[:,:,2*i:2*i+2]; xi=xc[:,2*i:2*i+2]; ki=jnp.sum(jnp.abs(xi)**2,axis=1,keepdims=True)
    JiJi=jnp.einsum('...ai,...bi->...ab',Ji,jnp.conj(Ji)); Jixi=jnp.einsum('...ai,...i->...a',Ji,xi)
    out=jnp.einsum('...a,...b->...ab',Jixi,jnp.conj(Jixi))
    gref=(ki[:,:,None]*JiJi - out)/(ki**2)[:,:,None]
    return jnp.real(jnp.trace(gi@gref,axis1=-2,axis2=-1))
CHUNK=512
hym_rho={n:np.zeros(n_train,np.float32) for n in hym_names}
J_all=np.zeros((n_train,3,8),np.complex64); ginv_all=np.zeros((n_train,3,3),np.complex64)
for s in tqdm(range(0,n_train,CHUNK)):
    e=min(s+CHUNK,n_train); Xc=jnp.array(X_train[s:e],jnp.float32)
    gi,J,xc=compute_chunk(Xc); tr=[gref_tr(gi,J,xc,i) for i in range(4)]
    for nm,kv in zip(hym_names,hym_charges):
        hym_rho[nm][s:e]=sum(float(kv[i])*np.array(tr[i]) for i in range(4)).astype(np.float32)
    J_all[s:e]=np.array(J); ginv_all[s:e]=np.array(gi)
print("HYM sources ready:", {n:round(float(hym_rho[n].mean()),4) for n in hym_names})

100%|██████████| 879/879 [00:22<00:00, 39.66it/s]

HYM sources ready: {'L5': 2.7169, 'fb': 0.306}


In [6]:
def M_to_omega(M):
    R=jnp.real(M)/4.; I=jnp.imag(M)/4.
    return jnp.concatenate([jnp.concatenate([R,-I],1), jnp.concatenate([I,R],1)],0)
def cy_laplacian(f, x, Om):
    gfn=jax.grad(f)
    HO=jax.vmap(lambda c: jax.jvp(gfn,(x,),(c,))[1], in_axes=1, out_axes=0)(Om)
    return jnp.trace(HO)
def make_hym_nn(sk):
    ks=jax.random.split(sk,5)
    return eqx.nn.Sequential([eqx.nn.Lambda(kappa_features),
        eqx.nn.Linear(16,128,key=ks[1]),eqx.nn.Lambda(jnn.gelu),
        eqx.nn.Linear(128,128,key=ks[2]),eqx.nn.Lambda(jnn.gelu),
        eqx.nn.Linear(128,128,key=ks[3]),eqx.nn.Lambda(jnn.gelu),
        eqx.nn.Linear(128,1,use_bias=False,key=ks[4])])
@eqx.filter_jit
def hym_step(model,Xb,Jb,gib,rhob,wb,opt_state,optimizer):
    def loss_fn(m_):
        def pp(x,Ji,gii,rho):
            M=jnp.conj(Ji).T@gii@Ji; lap=cy_laplacian(lambda z:m_(z)[0],x,M_to_omega(M))
            return jnp.abs(lap-rho)
        pw=jax.vmap(pp)(Xb,Jb,gib,rhob); return jnp.sum(wb*pw)/jnp.sum(wb)
    loss,gr=eqx.filter_value_and_grad(loss_fn)(model)
    upd,os_=optimizer.update(gr,opt_state,eqx.filter(model,eqx.is_array))
    return eqx.apply_updates(model,upd),os_,loss
w_all=y_train[:,0].astype(np.float32)
hym_models={}; hym_histories={}
for bi,nm in enumerate(hym_names):
    key,sk=jax.random.split(key); m=OrbitAvg(make_hym_nn(sk))
    opt=optax.adam(1e-3); os_=opt.init(eqx.filter(m,eqx.is_array))
    ref=float(np.sum(w_all*np.abs(hym_rho[nm]))/np.sum(w_all)); h=[]
    print(f"HYM {nm}:")
    for ep in range(25):
        perm=np.random.permutation(n_train); el=0.; nb=0
        for s in range(0,n_train,128):
            idx=perm[s:s+128]
            m,os_,loss=hym_step(m, jnp.array(X_train[idx],jnp.float32), jnp.array(J_all[idx],jnp.complex64),
                                jnp.array(ginv_all[idx],jnp.complex64), jnp.array(hym_rho[nm][idx],jnp.float32),
                                jnp.array(w_all[idx],jnp.float32), os_, opt)
            el+=float(loss); nb+=1
        h.append(el/nb/ref); print(f"  ep{ep+1:2d}: M_HYM={h[-1]:.4f}")
    hym_models[nm]=m; hym_histories[nm]=h

HYM L5:
  ep 1: M_HYM=0.5843
  ep 2: M_HYM=0.4075
  ep 3: M_HYM=0.2510
  ep 4: M_HYM=0.2143
  ep 5: M_HYM=0.1891
  ep 6: M_HYM=0.1740
  ep 7: M_HYM=0.1622
  ep 8: M_HYM=0.1544
  ep 9: M_HYM=0.1466
  ep10: M_HYM=0.1441
  ep11: M_HYM=0.1400
  ep12: M_HYM=0.1364
  ep13: M_HYM=0.1316
  ep14: M_HYM=0.1287
  ep15: M_HYM=0.1261
  ep16: M_HYM=0.1239
  ep17: M_HYM=0.1215
  ep18: M_HYM=0.1192
  ep19: M_HYM=0.1165
  ep20: M_HYM=0.1141
  ep21: M_HYM=0.1111
  ep22: M_HYM=0.1101
  ep23: M_HYM=0.1082
  ep24: M_HYM=0.1058
  ep25: M_HYM=0.1034
HYM fb:
  ep 1: M_HYM=0.5188
  ep 2: M_HYM=0.3192
  ep 3: M_HYM=0.2510
  ep 4: M_HYM=0.2245
  ep 5: M_HYM=0.2100
  ep 6: M_HYM=0.1971
  ep 7: M_HYM=0.1921
  ep 8: M_HYM=0.1833
  ep 9: M_HYM=0.1773
  ep10: M_HYM=0.1758
  ep11: M_HYM=0.1682
  ep12: M_HYM=0.1676
  ep13: M_HYM=0.1634
  ep14: M_HYM=0.1582
  ep15: M_HYM=0.1573
  ep16: M_HYM=0.1551
  ep17: M_HYM=0.1522
  ep18: M_HYM=0.1485
  ep19: M_HYM=0.1478
  ep20: M_HYM=0.1451
  ep21: M_HYM=0.1427
  ep22: M_HYM=0.13

## Reference forms — model-2 type-I (`L5`: 6 forms, `fb`: 2 forms), and their sources

In [7]:
def _mu_bar(x_cplx, j):
    v=jnp.zeros(8,jnp.complex64)
    v=v.at[2*j].set(-jnp.conj(x_cplx[2*j+1])); v=v.at[2*j+1].set(jnp.conj(x_cplx[2*j])); return v

# field -> (bundle, form params, eps).  eps = the form's g1-eigenvalue: (-1)^(a+b+1) for L5, (-1)^(b+1) for fb.
# (Wilson W=diag(1,1,1,-1,-1), rho_a=+1 then label each form as an MSSM field.)
FORM_SPEC = {
    'U1':('L5',(0,0),-1),'U2':('L5',(1,1),-1),'U3':('L5',(2,0),-1),     # u^c (=e^c)
    'Q1':('L5',(0,1),+1),'Q2':('L5',(1,0),+1),'Q3':('L5',(2,1),+1),     # Q
    'D' :('fb',(0,), -1),                                               # d^c
    'L' :('fb',(1,), +1),                                               # lepton doublet
}
form_names   = list(FORM_SPEC)
form_bundle  = {f:FORM_SPEC[f][0] for f in form_names}
form_eps     = {f:FORM_SPEC[f][2] for f in form_names}
form_charges = {f:(K_L5 if FORM_SPEC[f][0]=='L5' else K_FB) for f in form_names}

def nu_ref(x_cplx, form_name):
    bnd, prm, _ = FORM_SPEC[form_name]
    if bnd=='L5':
        a,b=prm; kap=jnp.sum(jnp.abs(x_cplx[6:8])**2)
        poly=x_cplx[0]**(2-a)*x_cplx[1]**a * x_cplx[2]**(1-b)*x_cplx[3]**b
        return (poly/kap**2)*_mu_bar(x_cplx,3), K_L5
    else:
        b=prm[0]; kap=jnp.sum(jnp.abs(x_cplx[4:6])**2)
        poly=x_cplx[2]**(1-b)*x_cplx[3]**b
        return (poly/kap**2)*_mu_bar(x_cplx,2), K_FB

def log_H_grad(x_cplx, k_vec):
    g=jnp.zeros(8,jnp.complex64)
    for i in range(4):
        xi=x_cplx[2*i:2*i+2]; kap=jnp.sum(jnp.abs(xi)**2)
        g=g.at[2*i:2*i+2].set(-k_vec[i]*jnp.conj(xi)/kap)
    return g
print("Reference forms (type-I) defined:", form_names)

Reference forms (type-I) defined: ['U1', 'U2', 'U3', 'Q1', 'Q2', 'Q3', 'D', 'L']


In [8]:
# Pre-compute sigma sources rho_nu (HYM bundle metric H = e^beta H_ref)
def make_HNu(fn):
    def f(xr):
        xc=(xr[:8]+1j*xr[8:]).astype(jnp.complex64); nu,kv=nu_ref(xc,fn)
        kap=jnp.array([jnp.sum(jnp.abs(xc[2*i:2*i+2])**2) for i in range(4)])
        H=jnp.prod(kap**(-jnp.array(kv)))*jnp.exp(hym_models[form_bundle[fn]](xr)[0])
        HNu=H*nu; return jnp.concatenate([jnp.real(HNu),jnp.imag(HNu)])
    return f
def make_H(fn):
    def f(xc):
        _,kv=nu_ref(xc,fn); kap=jnp.array([jnp.sum(jnp.abs(xc[2*i:2*i+2])**2) for i in range(4)])
        xr=jnp.concatenate([jnp.real(xc),jnp.imag(xc)])
        return jnp.prod(kap**(-jnp.array(kv)))*jnp.exp(hym_models[form_bundle[fn]](xr)[0])
    return f
def make_source(fn):
    HNu=make_HNu(fn); H=make_H(fn)
    @jax.jit
    def sb(Xc,gic,Jc):
        xc=(Xc[:,:8]+1j*Xc[:,8:]).astype(jnp.complex64)
        d=jax.vmap(jax.jacfwd(HNu))(Xc)
        dw=(0.5*(d[:,:8,:8]+d[:,8:,8:])+0.5j*(d[:,8:,:8]-d[:,:8,8:])).astype(jnp.complex64)
        T=jnp.einsum('xAb,xab,xBa->xAB', Jc, dw, jnp.conj(Jc))
        Hv=jax.vmap(H)(xc)
        return -jnp.trace(gic@T,axis1=-2,axis2=-1)/Hv
    return sb
source_fns={f:make_source(f) for f in form_names}
sigma_rho={f:np.zeros(n_train,np.complex64) for f in form_names}
print("Pre-computing sigma sources ...")
for s in tqdm(range(0,n_train,512)):
    e=min(s+512,n_train); Xc=jnp.array(X_train[s:e],jnp.float32)
    gic=jnp.array(ginv_all[s:e],jnp.complex64); Jc=jnp.array(J_all[s:e],jnp.complex64)
    for f in form_names: sigma_rho[f][s:e]=np.array(source_fns[f](Xc,gic,Jc))
print("sigma sources ready.")

Pre-computing sigma sources ...


100%|██████████| 879/879 [01:24<00:00, 10.43it/s]

sigma sources ready.


In [9]:
# Train the 8 harmonic-form (sigma) nets, projected onto the Z2 parity eps of each form
def n_sections(k):
    n=1
    for ki in k: n*=abs(ki)+1
    return n
def build_sections(xc,k):
    per=[]
    for i,ki in enumerate(k):
        xi=xc[2*i:2*i+2]; m=abs(int(ki))
        if ki>=0: segs=[xi[0]**(m-a)*xi[1]**a for a in range(m+1)]
        else:
            kap=jnp.sum(jnp.abs(xi)**2); segs=[jnp.conj(xi[0])**(m-a)*jnp.conj(xi[1])**a/kap**m for a in range(m+1)]
        per.append(jnp.stack(segs))
    r=per[0]
    for s in per[1:]: r=jnp.outer(r,s).reshape(-1)
    return r.astype(jnp.complex64)
def make_sigma_nn(sk,nout):
    ks=jax.random.split(sk,5)
    return eqx.nn.Sequential([eqx.nn.Lambda(kappa_features),
        eqx.nn.Linear(16,128,key=ks[1]),eqx.nn.Lambda(jnn.gelu),
        eqx.nn.Linear(128,128,key=ks[2]),eqx.nn.Lambda(jnn.gelu),
        eqx.nn.Linear(128,128,key=ks[3]),eqx.nn.Lambda(jnn.gelu),
        eqx.nn.Linear(128,nout,use_bias=False,key=ks[4])])
def make_sigma_step(k_tuple, eps, beta_model):
    N=n_sections(k_tuple); kj=jnp.array(k_tuple,jnp.float32); e1=jnp.float32(eps)
    def _raw(m_,x):
        xc=(x[:8]+1j*x[8:]).astype(jnp.complex64); s=build_sections(xc,k_tuple)
        out=m_(x); c=out[:N]+1j*out[N:]; return jnp.dot(c,s)
    def sig(m_,x): return 0.5*(_raw(m_,x)+e1*_raw(m_,_g1(x)))       # Z2 parity projection
    @eqx.filter_jit
    def step(model,Xb,Jb,gib,rhob,wb,opt_state,optimizer):
        def loss_fn(m_):
            def pp(x,Ji,gii,rho):
                xc=(x[:8]+1j*x[8:]).astype(jnp.complex64); M=jnp.conj(Ji).T@gii@Ji; Om=M_to_omega(M)
                dlnH=log_H_grad(xc,kj)
                gb=jax.grad(lambda z:beta_model(z)[0])(x); dlnH=dlnH+0.5*(gb[:8]-1j*gb[8:])
                v=jnp.conj(Ji).T@(gii@(Ji@dlnH))
                def sR(z): return jnp.real(sig(m_,z))
                def sI(z): return jnp.imag(sig(m_,z))
                gR=jax.grad(sR)(x); gI=jax.grad(sI)(x)
                db=0.5*((gR[:8]+1j*gR[8:])+1j*(gI[:8]+1j*gI[8:]))
                lap=cy_laplacian(sR,x,Om)+1j*cy_laplacian(sI,x,Om)
                return jnp.abs(lap+jnp.sum(v*db)-rho)
            pw=jax.vmap(pp)(Xb,Jb,gib,rhob); return jnp.sum(wb*pw)/jnp.sum(wb)
        loss,gr=eqx.filter_value_and_grad(loss_fn)(model)
        upd,os_=optimizer.update(gr,opt_state,eqx.filter(model,eqx.is_array))
        return eqx.apply_updates(model,upd),os_,loss
    return step

sigma_models={}; sigma_histories={}
for f in form_names:
    k_tuple=tuple(int(v) for v in form_charges[f]); N=n_sections(k_tuple)
    key,sk=jax.random.split(key); m=make_sigma_nn(sk,2*N)
    step=make_sigma_step(k_tuple, form_eps[f], hym_models[form_bundle[f]])
    opt=optax.adam(1e-3); os_=opt.init(eqx.filter(m,eqx.is_array))
    ref=float(np.sum(w_all*np.abs(sigma_rho[f]))/np.sum(w_all)); h=[]
    print(f"sigma {f} (bundle {form_bundle[f]}, eps={form_eps[f]:+d}):")
    for ep in range(25):
        perm=np.random.permutation(n_train); el=0.; nb=0
        for s in range(0,n_train,128):
            idx=perm[s:s+128]
            m,os_,loss=step(m, jnp.array(X_train[idx],jnp.float32), jnp.array(J_all[idx],jnp.complex64),
                            jnp.array(ginv_all[idx],jnp.complex64), jnp.array(sigma_rho[f][idx],jnp.complex64),
                            jnp.array(w_all[idx],jnp.float32), os_, opt)
            el+=float(loss); nb+=1
        h.append(el/nb/ref); print(f"  ep{ep+1:2d}: M_sigma={h[-1]:.4f}")
    sigma_models[f]=m; sigma_histories[f]=h

sigma U1 (bundle L5, eps=-1):
  ep 1: M_sigma=0.6453
  ep 2: M_sigma=0.5140
  ep 3: M_sigma=0.4757
  ep 4: M_sigma=0.4518
  ep 5: M_sigma=0.4121
  ep 6: M_sigma=0.3750
  ep 7: M_sigma=0.3590
  ep 8: M_sigma=0.3477
  ep 9: M_sigma=0.3369
  ep10: M_sigma=0.3259
  ep11: M_sigma=0.3172
  ep12: M_sigma=0.3074
  ep13: M_sigma=0.2994
  ep14: M_sigma=0.2928
  ep15: M_sigma=0.2884
  ep16: M_sigma=0.2845
  ep17: M_sigma=0.2784
  ep18: M_sigma=0.2762
  ep19: M_sigma=0.2727
  ep20: M_sigma=0.2698
  ep21: M_sigma=0.2667
  ep22: M_sigma=0.2635
  ep23: M_sigma=0.2611
  ep24: M_sigma=0.2597
  ep25: M_sigma=0.2575
sigma U2 (bundle L5, eps=-1):
  ep 1: M_sigma=0.6747
  ep 2: M_sigma=0.5557
  ep 3: M_sigma=0.5219
  ep 4: M_sigma=0.4957
  ep 5: M_sigma=0.4685
  ep 6: M_sigma=0.4421
  ep 7: M_sigma=0.4195
  ep 8: M_sigma=0.4043
  ep 9: M_sigma=0.3951
  ep10: M_sigma=0.3853
  ep11: M_sigma=0.3771
  ep12: M_sigma=0.3683
  ep13: M_sigma=0.3617
  ep14: M_sigma=0.3539
  ep15: M_sigma=0.3485
  ep16: M_sigma=0.34

## Save the trained networks and point cloud (`model2_run_*.pkl`)

In [10]:
timestamp=datetime.datetime.now().strftime('%Y%m%d_%H%M%S')
fn=f'model2_run_{timestamp}.pkl'
def to_np(m): return jax.tree_util.tree_map(lambda x: np.asarray(x) if hasattr(x,'shape') else x, m)
_full=np.load(os.path.join(dirname,'dataset.npz'))
val_pb=_full['val_pullbacks'] if 'val_pullbacks' in _full else None
save={
  'metadata':{'timestamp':timestamp,'psi0':float(psi0),'psi1':float(psi1),
              'kmoduli':kmoduli.tolist(),'ambient':ambient.tolist(),'dirname':dirname,
              'n_train':int(n_train),'K_L5':K_L5.tolist(),'K_FB':K_FB.tolist(),
              'hym_names':hym_names,'form_names':form_names,
              'form_charges':{k:v.tolist() for k,v in form_charges.items()},
              'form_bundle':form_bundle,'form_eps':form_eps,
              'monomials':monomials.tolist(),'coefficients':coefficients.tolist()},
  'models':{'metric_model':to_np(metric_model),
            'hym_models':{k:to_np(v) for k,v in hym_models.items()},
            'sigma_models':{k:to_np(v) for k,v in sigma_models.items()}},
  'training_data':{'X_train':np.asarray(X_train),'y_train':np.asarray(y_train),
                   'X_val':np.asarray(X_val),'y_val':np.asarray(y_val),'val_pullbacks':val_pb},
  'precomputed':{'J_all':J_all,'ginv_all':ginv_all,
                 'hym_rho':{k:v for k,v in hym_rho.items()},
                 'sigma_rho':{k:v for k,v in sigma_rho.items()}},
  'histories':{'hym_histories':hym_histories,'sigma_histories':sigma_histories},
  'BASIS_raw':BASIS_raw,
}
with open(fn,'wb') as f:
    cd=dict(save); cd['models']=dict(save['models'])
    cd['models']['metric_model']=eqx.tree_at(lambda m:m._sigma_loss_fn, cd['models']['metric_model'], replace=[])
    pickle.dump(cd, f, protocol=pickle.HIGHEST_PROTOCOL)
print("Saved", fn, "(%.1f MB)"%(os.path.getsize(fn)/1024**2))
print("  metric @ kmoduli =", kmoduli, " | HYM:", list(hym_models), " | sigma:", list(sigma_models))

Saved model2_run_20260813_234318.pkl (232.8 MB)
  metric @ kmoduli = [4.         1.         3.         1.33333333]  | HYM: ['L5', 'fb']  | sigma: ['U1', 'U2', 'U3', 'Q1', 'Q2', 'Q3', 'D', 'L']
